# Reproduce the Manglish ATE experiments

This notebook executes the repository workflow from checked-in source, configuration,
resources and split manifests.

**Reproducibility rules**

- Experiment paths are repository-relative.
- No manuscript metric, expected score, support count, target mean, winner or preferred outcome is an input.
- `outputs/reference_run/` is verification material only and is never consumed by this notebook.
- The proposed pipeline predictions are regenerated before evaluation.
- Supervised models are trained using manifest-defined training/validation membership.
- The common final test set is read only for final evaluation.
- Full-supervision and low-resource regimes are both retained.

## 1. Clone/update the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/SreeBhagya-S/Low-Resource-Aspect-Term-Extraction.git"
REPO_DIR = Path("/content/Low-Resource-Aspect-Term-Extraction")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

print("Repository:", REPO_DIR)
print("Commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip())

## 2. Install the declared environment

In [ ]:
requirements = REPO_DIR / "requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(
        "requirements.txt is missing from the repository root."
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
    check=True
)

print("Environment installation complete.")

In [ ]:
import torch
import pandas as pd
import numpy as np
import json

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Optional Hugging Face authentication

If a model requires authentication, store a token in the Colab secret `HF_TOKEN`.
The token is never written to the repository. If all model checkpoints are publicly
accessible, this cell simply continues without logging in.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("No HF_TOKEN supplied; continuing with public model access.")

## 3. Repository paths and preflight

In [ ]:
ROOT = REPO_DIR
CODE = ROOT / "code"
CONFIG = ROOT / "config"
DATA = ROOT / "data"
RESOURCES = ROOT / "resources"
OUTPUT = ROOT / "outputs" / "current_run"

PROTOCOL_FILE = CONFIG / "protocol.json"

required_files = [
    CODE / "ate_pipeline_core.py",
    CODE / "generate_ate_bio.py",
    CODE / "evaluate_ate.py",
    CODE / "compare_methods.py",
    CODE / "train_transformer_baselines.py",
    CODE / "train_char_bilstm_crf.py",
    CODE / "ablation_study.py",

    PROTOCOL_FILE,
    CONFIG / "pipeline_config.json",
    CONFIG / "transformer_config.json",
    CONFIG / "char_bilstm_config.json",

    DATA / "evaluation_input.csv",
    DATA / "evaluation_gold.csv",
    DATA / "dual_regime_protocol_manifest.csv",
    DATA / "fixed_test_manifest_seed42.csv",

    RESOURCES / "aspect_lexicon.txt",
    RESOURCES / "merged_stopwords.txt",
    RESOURCES / "malayalam_suffix_1.txt",
    RESOURCES / "malayalam_suffixmorph.txt",
    RESOURCES / "positive-words_english.txt",
    RESOURCES / "negative-words_english.txt",
    RESOURCES / "pos_words_malayalam.txt",
    RESOURCES / "neg_word_malayalam.txt",
]

missing = [str(p.relative_to(ROOT)) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Repository preflight failed. Missing files:\n  - "
        + "\n  - ".join(missing)
    )

print("PASS: all required repository files are present.")

## 4. Validate protocol and fixed split manifests

In [ ]:
protocol = json.loads(PROTOCOL_FILE.read_text(encoding="utf-8"))

# Evaluation outcomes are not valid protocol inputs.
forbidden_protocol_keys = {
    "test_gold_support",
    "expected_f1",
    "expected_score",
    "target_f1",
    "target_mean",
    "preferred_outcome",
    "winner"
}

def walk_keys(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            yield str(key)
            yield from walk_keys(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk_keys(value)

present = sorted({
    key for key in walk_keys(protocol)
    if key.lower() in forbidden_protocol_keys
})

if present:
    raise AssertionError(
        "Outcome-derived protocol keys found: " + ", ".join(present)
    )

print("PASS: protocol contains experimental design only.")

In [ ]:
protocol_manifest = pd.read_csv(DATA / "dual_regime_protocol_manifest.csv")
test_manifest = pd.read_csv(DATA / "fixed_test_manifest_seed42.csv")

expected_columns = {
    "Original_Row_Index",
    "Outer_Role",
    "Full_Supervised_Role",
    "Low_Resource_Role",
    "Pipeline_Role"
}

missing_columns = expected_columns - set(protocol_manifest.columns)
if missing_columns:
    raise KeyError(
        "Protocol manifest missing columns: "
        + ", ".join(sorted(missing_columns))
    )

outer = protocol["split_assignment"]
full = protocol["supervised_regimes"]["full_supervision"]
low = protocol["supervised_regimes"]["low_resource"]
pipe = protocol["proposed_pipeline"]

assert (protocol_manifest["Outer_Role"] == "development").sum() == outer["development_reviews"]
assert (protocol_manifest["Outer_Role"] == "final_test").sum() == outer["final_test_reviews"]

assert (protocol_manifest["Full_Supervised_Role"] == "train").sum() == full["training_reviews"]
assert (protocol_manifest["Full_Supervised_Role"] == "validation").sum() == full["validation_reviews"]
assert (protocol_manifest["Full_Supervised_Role"] == "test").sum() == outer["final_test_reviews"]

assert (protocol_manifest["Low_Resource_Role"] == "train").sum() == low["training_reviews"]
assert (protocol_manifest["Low_Resource_Role"] == "validation").sum() == low["validation_reviews"]
assert (protocol_manifest["Low_Resource_Role"] == "test").sum() == outer["final_test_reviews"]

assert (protocol_manifest["Pipeline_Role"] == "configuration").sum() == pipe["configuration_reviews"]
assert (protocol_manifest["Pipeline_Role"] == "test").sum() == outer["final_test_reviews"]

outer_test = set(
    protocol_manifest.loc[
        protocol_manifest["Outer_Role"] == "final_test",
        "Original_Row_Index"
    ].astype(int)
)

fixed_test = set(test_manifest["Original_Row_Index"].astype(int))

assert outer_test == fixed_test, (
    "Fixed test manifest and outer final-test assignment differ."
)

test_rows = protocol_manifest[
    protocol_manifest["Outer_Role"] == "final_test"
]

assert test_rows["Full_Supervised_Role"].eq("test").all()
assert test_rows["Low_Resource_Role"].eq("test").all()
assert test_rows["Pipeline_Role"].eq("test").all()

print("PASS: fixed split membership is internally consistent.")
print("Development reviews:", outer["development_reviews"])
print("Final test reviews:", outer["final_test_reviews"])
print("Full supervision:", full["training_reviews"], "train +", full["validation_reviews"], "validation")
print("Low resource:", low["training_reviews"], "train +", low["validation_reviews"], "validation")
print("Pipeline configuration:", pipe["configuration_reviews"], "reviews")

## 5. Verify repository input dimensions

In [ ]:
gold_df = pd.read_csv(DATA / "evaluation_gold.csv")
input_df = pd.read_csv(DATA / "evaluation_input.csv")

if len(gold_df) != protocol["corpus"]["annotated_reviews"]:
    raise AssertionError(
        f"Gold file has {len(gold_df)} reviews; protocol specifies "
        f"{protocol['corpus']['annotated_reviews']}."
    )

if len(input_df) != protocol["corpus"]["annotated_reviews"]:
    raise AssertionError(
        f"Pipeline input has {len(input_df)} reviews; protocol specifies "
        f"{protocol['corpus']['annotated_reviews']}."
    )

if len(test_manifest) != protocol["split_assignment"]["final_test_reviews"]:
    raise AssertionError("Unexpected test-manifest size.")

print("PASS: input, gold and test-manifest sizes are consistent.")

## 6. Build runtime training/validation manifests

In [ ]:
RUNTIME_MANIFESTS = OUTPUT / "manifests"
RUNTIME_MANIFESTS.mkdir(parents=True, exist_ok=True)

def make_training_manifest(role_column, destination):
    roles = protocol_manifest[["Original_Row_Index", role_column]].copy()
    roles["Role"] = roles[role_column].map({
        "train": "supervised_train",
        "validation": "supervised_validation"
    }).fillna("not_used")
    roles[["Original_Row_Index", "Role"]].to_csv(destination, index=False)
    return destination

FULL_SPLIT = make_training_manifest(
    "Full_Supervised_Role",
    RUNTIME_MANIFESTS / "full_supervision_split.csv"
)

LOW_SPLIT = make_training_manifest(
    "Low_Resource_Role",
    RUNTIME_MANIFESTS / "low_resource_split.csv"
)

print("Full-supervision split:", FULL_SPLIT)
print("Low-resource split:", LOW_SPLIT)

## 7. Start a clean current run

Only `outputs/current_run/` is cleared. `outputs/reference_run/`, if present,
is never read by the execution workflow.

In [ ]:
import shutil

OUTPUT.mkdir(parents=True, exist_ok=True)

for child in list(OUTPUT.iterdir()):
    if child.name == "manifests":
        continue
    if child.is_dir():
        shutil.rmtree(child)
    else:
        child.unlink()

print("Current run:", OUTPUT)

## 8. Regenerate proposed-pipeline predictions

In [ ]:
PIPELINE_OUT = OUTPUT / "pipeline"
PIPELINE_OUT.mkdir(parents=True, exist_ok=True)

pipeline_predictions = PIPELINE_OUT / "pipeline_predictions.csv"
pipeline_diagnostics = PIPELINE_OUT / "candidate_diagnostics.csv"

cmd = [
    sys.executable, str(CODE / "generate_ate_bio.py"),
    "--config", str(CONFIG / "pipeline_config.json"),
    "--input", str(DATA / "evaluation_input.csv"),
    "--output", str(pipeline_predictions),
    "--lexicon", str(RESOURCES / "aspect_lexicon.txt"),
    "--suffix-file", str(RESOURCES / "malayalam_suffix_1.txt"),
    "--suffix-file", str(RESOURCES / "malayalam_suffixmorph.txt"),
    "--stopwords", str(RESOURCES / "merged_stopwords.txt"),
    "--sentiment-file", str(RESOURCES / "neg_word_malayalam.txt"),
    "--sentiment-file", str(RESOURCES / "negative-words_english.txt"),
    "--sentiment-file", str(RESOURCES / "pos_words_malayalam.txt"),
    "--sentiment-file", str(RESOURCES / "positive-words_english.txt"),
    "--diagnostics", str(pipeline_diagnostics),
]

if torch.cuda.is_available():
    cmd += ["--device", "cuda"]

subprocess.run(cmd, check=True)

generated = pd.read_csv(pipeline_predictions)
assert len(generated) == protocol["corpus"]["annotated_reviews"]

print("Generated:", pipeline_predictions)

## 9. Evaluate the regenerated pipeline on the fixed final test set

In [ ]:
PIPELINE_EVAL = PIPELINE_OUT / "evaluation"

subprocess.run([
    sys.executable, str(CODE / "evaluate_ate.py"),
    "--gold", str(DATA / "evaluation_gold.csv"),
    "--pred", str(pipeline_predictions),
    "--manifest", str(DATA / "fixed_test_manifest_seed42.csv"),
    "--output-dir", str(PIPELINE_EVAL),
    "--system-name", "Proposed weakly supervised ATE",
], check=True)

display(pd.read_csv(PIPELINE_EVAL / "strict_span_metrics.csv"))

## 10. Train supervised transformer baselines in both regimes

In [ ]:
TRANSFORMER_CONFIG = CONFIG / "transformer_config.json"

FULL_TRANSFORMERS = OUTPUT / "full_supervision" / "transformers"
LOW_TRANSFORMERS = OUTPUT / "low_resource" / "transformers"

for split_file, out_dir, label in [
    (FULL_SPLIT, FULL_TRANSFORMERS, "full supervision"),
    (LOW_SPLIT, LOW_TRANSFORMERS, "low resource"),
]:
    print("\nRunning transformer baselines:", label)
    subprocess.run([
        sys.executable, str(CODE / "train_transformer_baselines.py"),
        "--gold", str(DATA / "evaluation_gold.csv"),
        "--split-manifest", str(split_file),
        "--train-role", "supervised_train",
        "--validation-role", "supervised_validation",
        "--test-manifest", str(DATA / "fixed_test_manifest_seed42.csv"),
        "--config", str(TRANSFORMER_CONFIG),
        "--output-dir", str(out_dir),
    ], check=True)

## 11. Train Char-BiLSTM+CRF in both regimes

In [ ]:
CHAR_CONFIG = CONFIG / "char_bilstm_config.json"

FULL_BILSTM = OUTPUT / "full_supervision" / "bilstm"
LOW_BILSTM = OUTPUT / "low_resource" / "bilstm"

for split_file, out_dir, label in [
    (FULL_SPLIT, FULL_BILSTM, "full supervision"),
    (LOW_SPLIT, LOW_BILSTM, "low resource"),
]:
    print("\nRunning Char-BiLSTM+CRF:", label)
    subprocess.run([
        sys.executable, str(CODE / "train_char_bilstm_crf.py"),
        "--gold", str(DATA / "evaluation_gold.csv"),
        "--split-manifest", str(split_file),
        "--train-role", "supervised_train",
        "--validation-role", "supervised_validation",
        "--test-manifest", str(DATA / "fixed_test_manifest_seed42.csv"),
        "--config", str(CHAR_CONFIG),
        "--output-dir", str(out_dir),
    ], check=True)

## 12. Recompute comparisons from generated prediction files

In [ ]:
def compare_regime(name, transformer_dir, bilstm_dir):
    destination = OUTPUT / "comparison" / name
    destination.mkdir(parents=True, exist_ok=True)

    systems = [
        ("Proposed weakly supervised ATE", pipeline_predictions),
        ("mBERT", transformer_dir / "mbert" / "word_level_predictions.csv"),
        ("IndicBERT", transformer_dir / "indicbert" / "word_level_predictions.csv"),
        ("XLM-R", transformer_dir / "xlmr" / "word_level_predictions.csv"),
        ("Char-BiLSTM+CRF", bilstm_dir / "word_level_predictions.csv"),
    ]

    cmd = [
        sys.executable, str(CODE / "compare_methods.py"),
        "--gold", str(DATA / "evaluation_gold.csv"),
        "--manifest", str(DATA / "fixed_test_manifest_seed42.csv"),
    ]

    for system_name, prediction_file in systems:
        if not prediction_file.exists():
            raise FileNotFoundError(prediction_file)
        cmd += ["--system", f"{system_name}={prediction_file}"]

    cmd += ["--output-dir", str(destination)]
    subprocess.run(cmd, check=True)

    return pd.read_csv(destination / "method_comparison_micro.csv")

full_comparison = compare_regime(
    "full_supervision", FULL_TRANSFORMERS, FULL_BILSTM
)
low_comparison = compare_regime(
    "low_resource", LOW_TRANSFORMERS, LOW_BILSTM
)

print("Full supervision")
display(full_comparison)

print("Low resource")
display(low_comparison)

## 13. Component ablation

In [ ]:
ABLATION_OUT = OUTPUT / "ablation"

cmd = [
    sys.executable, str(CODE / "ablation_study.py"),
    "--generator", str(CODE / "generate_ate_bio.py"),
    "--config", str(CONFIG / "pipeline_config.json"),
    "--input", str(DATA / "evaluation_input.csv"),
    "--gold", str(DATA / "evaluation_gold.csv"),
    "--manifest", str(DATA / "fixed_test_manifest_seed42.csv"),
    "--lexicon", str(RESOURCES / "aspect_lexicon.txt"),
    "--suffix-file", str(RESOURCES / "malayalam_suffix_1.txt"),
    "--suffix-file", str(RESOURCES / "malayalam_suffixmorph.txt"),
    "--stopwords", str(RESOURCES / "merged_stopwords.txt"),
    "--sentiment-file", str(RESOURCES / "neg_word_malayalam.txt"),
    "--sentiment-file", str(RESOURCES / "negative-words_english.txt"),
    "--sentiment-file", str(RESOURCES / "pos_words_malayalam.txt"),
    "--sentiment-file", str(RESOURCES / "positive-words_english.txt"),
    "--output-dir", str(ABLATION_OUT),
]

if torch.cuda.is_available():
    cmd += ["--device", "cuda"]

subprocess.run(cmd, check=True)
display(pd.read_csv(ABLATION_OUT / "component_ablation.csv"))

## 14. Audit executable source and configuration before publication

In [ ]:
import re

audit_patterns = {
    "machine-specific path":
        r"/content/drive|MyDrive|/home/[^/]+|[A-Za-z]:\\\\",
    "outcome-derived input wording":
        r"expected[_ -]?(f1|score|metric)|"
        r"target[_ -]?(mean|f1|score)|"
        r"preferred[_ -]?(outcome|score)|"
        r"test_gold_support|"
        r"predefined[_ -]?target",
    "obsolete artificial-result wording":
        r"artificial[_ -]?run|fabricated[_ -]?score",
}

issues = []

# The notebook is excluded from this grep because it contains the audit expressions
# themselves. Audit the executable source and configuration that determine a run.
for base in [CODE, CONFIG]:
    for path in base.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".py", ".json"}:
            continue

        source = path.read_text(encoding="utf-8", errors="ignore")

        for label, pattern in audit_patterns.items():
            if re.search(pattern, source, flags=re.I):
                issues.append((str(path.relative_to(ROOT)), label))

if issues:
    print("REVIEW THESE MATCHES:")
    for path, issue in issues:
        print(f"{issue:34s} {path}")
else:
    print("PASS: no flagged source/config patterns found.")

print(
    "\nA match is a review prompt, not an automatic deletion instruction. "
    "For example, a candidate-category score inside the ATE algorithm is not "
    "an evaluation target."
)

## 15. Completion check

In [ ]:
expected_outputs = [
    pipeline_predictions,
    PIPELINE_EVAL / "strict_span_metrics.csv",

    FULL_TRANSFORMERS / "mbert" / "word_level_predictions.csv",
    FULL_TRANSFORMERS / "indicbert" / "word_level_predictions.csv",
    FULL_TRANSFORMERS / "xlmr" / "word_level_predictions.csv",
    FULL_BILSTM / "word_level_predictions.csv",

    LOW_TRANSFORMERS / "mbert" / "word_level_predictions.csv",
    LOW_TRANSFORMERS / "indicbert" / "word_level_predictions.csv",
    LOW_TRANSFORMERS / "xlmr" / "word_level_predictions.csv",
    LOW_BILSTM / "word_level_predictions.csv",

    OUTPUT / "comparison" / "full_supervision" / "method_comparison_micro.csv",
    OUTPUT / "comparison" / "low_resource" / "method_comparison_micro.csv",
    ABLATION_OUT / "component_ablation.csv",
]

missing_outputs = [
    str(path.relative_to(ROOT))
    for path in expected_outputs
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Run incomplete. Missing outputs:\n  - "
        + "\n  - ".join(missing_outputs)
    )

print("PASS: all required current-run artifacts were generated.")

## 16. Before committing the reference run

- Compare regenerated outputs with the manuscript and response letter.
- Do not edit generated metrics manually.
- If regenerated values differ, resolve the experimental discrepancy and update the manuscript from the verified run.
- Keep `outputs/current_run/` disposable during development.
- After verification, copy only the accepted final artifacts to `outputs/reference_run/`.
- Commit and tag the exact code, configs, manifests and reference outputs used for the revision.